In [3]:
import pandas as pd
import os
import shutil

# --- 1. DEFINE YOUR PATHS ---
# Update these paths to match where you unzipped CrisisMMD
annotations_file = "crisismmd_v2.0/annotations/crisismmd_master.tsv"
raw_images_folder = "CrisisMMD_v2.0/"

# The folders we are creating for Teammate 2
target_vision_folder = "image_data/"
disaster_folder = os.path.join(target_vision_folder, "disaster")
safe_folder = os.path.join(target_vision_folder, "safe")

# Create the folders if they don't exist
os.makedirs(disaster_folder, exist_ok=True)
os.makedirs(safe_folder, exist_ok=True)

# --- 2. LOAD THE CRISISMMD ANNOTATIONS ---
# Note: It's a .tsv (Tab-Separated Values), not a comma-separated CSV!
print("Loading CrisisMMD annotations...")
df = pd.read_csv(annotations_file, sep='\t')

# --- 3. PREPARE TEXT FOR TEAMMATE 1 ---
print("Preparing Text CSV for Model 1...")
# We map 'informative' to 1 (Disaster) and 'not_informative' to 0 (Safe)
text_df = df[['tweet_text', 'text_info']].copy()
text_df = text_df.rename(columns={'tweet_text': 'text'})
text_df['label'] = text_df['text_info'].apply(lambda x: 1 if x == 'informative' else 0)

# Save the clean CSV for the DistilBERT model
text_df[['text', 'label']].to_csv('clean_crisismmd_tweets.csv', index=False)
print("Saved 'clean_crisismmd_tweets.csv' successfully!")

# --- 4. SORT IMAGES FOR TEAMMATE 2 ---
print("Sorting Images into folders for Model 2. This might take a minute...")

# Keep track of missing images just in case
missing_images = 0
copied_images = 0

for index, row in df.iterrows():
    # CrisisMMD gives us the relative path (e.g., 'california_wildfires/12345.jpg')
    img_path_in_tsv = str(row['image_path'])
    full_source_path = os.path.join(raw_images_folder, img_path_in_tsv)

    # Check the label for the image (Task 1)
    label = row['image_info']

    # Determine the destination folder
    if label == 'informative':
        destination_folder = disaster_folder
    elif label == 'not_informative':
        destination_folder = safe_folder
    else:
        continue # Skip if label is weird or missing

    # Get just the filename to avoid folder structure issues in the destination
    filename = os.path.basename(img_path_in_tsv)
    full_dest_path = os.path.join(destination_folder, filename)

    # Copy the file
    if os.path.exists(full_source_path):
        # We use try/except to prevent the loop from crashing if a file is corrupted
        try:
            shutil.copy2(full_source_path, full_dest_path)
            copied_images += 1
        except Exception as e:
            pass
    else:
        missing_images += 1

print(f"Done! Copied {copied_images} images.")
if missing_images > 0:
    print(f"Note: Could not find {missing_images} images listed in the TSV.")

Loading CrisisMMD annotations...
Preparing Text CSV for Model 1...
Saved 'clean_crisismmd_tweets.csv' successfully!
Sorting Images into folders for Model 2. This might take a minute...
Done! Copied 18082 images.
